
# Generación del conjunto de datos de entrenamiento

**Autor**: Juana María Rodríguez García

El objetivo de esta libreta es construir el conjunto de datos de entrenamiento para el modelo de detección de fraude en seguros de automóvil. Para ello, se combina la tabla `gold_fraud_spine` (que contiene el esqueleto de eventos etiquetados) con las dos tablas de características de la capa `Gold` (`gold_policy_profile` y `gold_policy_aggregations`), usando la `API` del `Feature Store` de `Databricks`.

El resultado final se guarda como tabla `Delta` estática en `Unity Catalog` bajo el nombre `gold_fraud_training_dataset`. Esta tabla cumple tres funciones críticas en la arquitectura:

* **Conjunto de datos de entrenamiento reproducible**: el cruce temporal entre la *spine* y las tablas de características (la operación más costosa del *pipeline*) se realiza una única vez. Los experimentos posteriores leen directamente desde esta tabla `Delta`, sin recalcular nada.
* **Fotografía congelada de los datos**: las tablas `Gold` son «vivas» y se actualizan cada hora. Al guardar el conjunto de entrenamiento en `Delta`, la versión exacta de los datos que vio el modelo queda registrada de forma inmutable, lo que permite reproducir cualquier entrenamiento meses después.
* **Referencia de *baseline* para monitorización**: esta misma tabla se usará en las libretas de monitorización como perfil de referencia para detectar *data drift* cuando el modelo esté en producción.

Esta libreta **no contiene lógica de transformación propia**. Todo el trabajo de cruce y enriquecimiento lo realiza internamente `create_training_set`, que delega en `Spark` la ejecución del *point-in-time* (`PiT`) *join* de forma distribuida.


## 1. Importación de librerías y configuración

La primera celda instala el paquete `databricks-feature-engineering`, necesario para acceder a `create_training_set` y a los objetos `FeatureLookup`.

A continuación se importan los componentes necesarios y se definen los nombres completamente cualificados de todas las tablas involucradas:

* `gold_fraud_spine`: la tabla de partida del cruce. Contiene el `customer_id`, el `timestamp` de cada transacción, los detalles de la misma y la etiqueta `is_fraud`.
* `gold_policy_profile`: tabla de características estáticas del cliente (edad, país, segmento, etc.).
* `gold_policy_aggregations`: tabla de características comportamentales calculadas con ventana deslizante (número de transacciones en los últimos 7 días, importe medio, etc.).
* `gold_fraud_training_dataset`: la tabla de salida donde se guardará el conjunto de entrenamiento enriquecido.

También se inicializa el cliente `FeatureEngineeringClient`.

In [0]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from datetime import datetime, timezone
from pyspark.sql.functions import col, count, round, when

In [0]:
catalog = "workspace"
database = "fraude-seguros-automovil"
gold_spine_table = f"{catalog}.`{database}`.gold_fraud_spine"
gold_policy_profile_table = f"{catalog}.{database}.gold_policy_profile"
gold_policy_aggregations_table = f"{catalog}.{database}.gold_policy_aggregations"

gold_training_dataset_table = f"{catalog}.`{database}`.gold_fraud_training_dataset"

fe = FeatureEngineeringClient()


## 2. Carga de la *spine*

La *spine* es el punto de partida de todo el proceso. Es la tabla que determina **qué filas** compondrán el conjunto de entrenamiento y **en qué instante** se evalúa cada cruce. Sin la *spine*, la `API` `create_training_set` no sabe ni cuántas filas debe generar ni qué marcas temporales debe usar para el `PiT` *join*.

La *spine* debe contener obligatoriamente:

* La clave de unión con las tablas de características: en nuestro caso, `policy_id`.
* La marca temporal del evento: `timestamp`. Esta columna es la que se pasa como `timestamp_lookup_key` en los `FeatureLookup` y define el `AS OF` de cada cruce.
* La etiqueta de supervisión: `is_fraud`. Esta columna es la variable objetivo que el modelo deberá aprender a predecir.

Se muestra el esquema y una muestra de la *spine* para verificar que tiene la estructura esperada antes de lanzar el cruce.

In [0]:
spine_df = spark.table(gold_spine_table)

print(f"Spine rows: {spine_df.count():,}")
print(f"Spine columns: {len(spine_df.columns)}")
spine_df.printSchema()


## 3. Definición de los `FeatureLookup`

Un `FeatureLookup` es el objeto que le indica a `create_training_set` cómo enriquecer cada fila de la *spine* con características procedentes de una tabla del `Feature Store`. Por cada tabla de características que queramos incorporar, se crea un `FeatureLookup` independiente.

Cada `FeatureLookup` tiene cuatro parámetros fundamentales:

* **`table_name`**: nombre completamente cualificado de la tabla de características en `Unity Catalog`.
* **`feature_names`**: lista de columnas de esa tabla que se quieren añadir al conjunto de datos de entrenamiento. Si se omite este parámetro, se incorporan todas las columnas de la tabla.
* **`lookup_key`**: columna o lista de columnas que se usa como clave de unión primaria entre la *spine* y la tabla de características. En nuestro caso, `policy_id`.
* **`timestamp_lookup_key`**: columna de la *spine* que actúa como referencia temporal. Aquí es donde se ejecuta el `PiT` *join*: se buscará, para cada fila de la *spine*, el registro más reciente de la tabla de características cuyo *timestamp* sea **anterior o igual** al valor de esta columna. Esto garantiza el comportamiento `AS OF` y previene cualquier fuga de datos del futuro (*data leakage*).

> **Nota arquitectónica clave**: Observa en el código posterior que no es necesario indicar a `FeatureLookup` cómo se llama la columna temporal en la tabla de características (por ejemplo, `__START_AT` en `gold_policy_profile`). El `Feature Store` lo sabe automáticamente porque definimos esa columna con el modificador `TIMESERIES` en la clave primaria al crear las tablas `Gold`.

A continuación, definimos dos `FeatureLookup`, uno por cada tabla de características de la capa `Gold`.

In [0]:
from databricks.feature_engineering import FeatureLookup

# 1. Claves de unión
entity_key = "policy_id"
timestamp_key = "timestamp"

# 2. Lista de características de perfil para Seguros de Automóvil
profile_feature_names = [
    # Demografía del Asegurado 
    "age_group",
    "vehicle_segment",
    "risk_level_by_premium",
    "driver_experience_years", 
    "is_new_policy_risk",
    "gender",
    "region_type",
    "coverage_type",           
    "vehicle_type"             

]

# 3. Definición del Lookup para el Feature Store
profile_lookup = FeatureLookup(
    table_name = gold_policy_profile_table,    
    feature_names = profile_feature_names,
    lookup_key = entity_key,
    timestamp_lookup_key = timestamp_key
)


# 3. Agregaciones de Comportamiento (Rolling Windows)
aggregation_feature_names = [
    # Ventana 24 horas
    "count_claims_24h",
    "sum_amount_24h",
    
    # Ventana 7 días
    "count_claims_7d",
    "sum_amount_7d",
    "avg_amount_7d",
    
    # Ventana 30 días
    "count_claims_30d",
    "sum_amount_30d",
    "avg_amount_30d",
    "max_amount_30d",
    "num_fraud_confirmed_30d",
    
    # Ratios e indicadores
    "claims_7d_vs_avg_30d_ratio"
]

aggregations_lookup = FeatureLookup(
    table_name = gold_policy_aggregations_table, 
    feature_names = aggregation_feature_names,
    lookup_key = entity_key,
    timestamp_lookup_key = timestamp_key
)

# 4. Lista final para generar el Training Set
feature_lookups = [profile_lookup, aggregations_lookup]

print(f"Características de perfil: {len(profile_feature_names)}")
print(f"Características de agregación: {len(aggregation_feature_names)}")
print(f"Total de variables añadidas: {len(profile_feature_names) + len(aggregation_feature_names)}")


## 4. Creación del conjunto de datos de entrenamiento con `create_training_set`

`fe.create_training_set` es la función central de esta libreta. Recibe la *spine*, la lista de `FeatureLookup` y el nombre de la columna que contiene la etiqueta de supervisión, y devuelve un objeto `TrainingSet`.

El `TrainingSet` es un objeto lógico: en este punto todavía no se ha ejecutado ningún cálculo. La `API` construye internamente el plan de ejecución del `PiT` *join*, pero la materialización real en un `DataFrame` de `Spark` no ocurre hasta que se llama a `.load_df()` en la siguiente celda. Esta distinción es relevante porque permite inspeccionar el plan antes de ejecutarlo.

El parámetro `label` le indica a `create_training_set` cuál es la columna objetivo. Esta información queda registrada en los metadatos del `TrainingSet` para que el sistema identifique qué variable es la que se debe predecir.

El parámetro `exclude_columns` permite eliminar columnas que no deben formar parte del conjunto de entrenamiento. En nuestro caso, se excluye `label_available_date` porque es un dato operativo que no se recibiría en una transacción cruda cuando llegue a producción. Descartar este tipo de columnas garantiza que el modelo solo aprenda de la información que verdaderamente tendrá disponible durante la inferencia en tiempo real.

In [0]:
# The column the model must learn to predict.
label = "is_fraud"

# Columns that must be excluded from the final training dataset are those
# that would not be received in a raw transaction when it arrives in production.
# Dropping them ensures the model only trains on data that will actually be
# available during real-time inference.
exclude_columns=[ "label_available_timestamp"] 
# Build the training dataset logical plan. The following line does not trigger
# any computation. Instead, it returns an object that encodes the spine dataset
# to start from, the feature lookups to join along with their point-in-time
# semantics, the column acting as the label, and the specific columns to drop
# before materialization.

training_dataset = fe.create_training_set(
    df = spine_df,
    feature_lookups = feature_lookups,
    label = label,
    exclude_columns = exclude_columns
)

print("Training dataset logical plan created, but no data materialized yet.")
print(f"Label column: {label}")
print(f"Excluded columns: {exclude_columns}")

In [0]:
# Materialize the training dataset.
# This is the step that actually executes the joins across the cluster.
training_df = training_dataset.load_df()
# 1. Eliminamos lo que no tiene etiqueta (no podemos entrenar sin saber la verdad)
# 2. Eliminamos filas donde no hay perfil (el join falló por fechas)

# Verify the result: total rows and final column set
print(f"Training dataset rows: {training_df.count():,}")
print(f"Training dataset columns: {len(training_df.columns)}")
training_df.printSchema()

In [0]:
# Quick visual inspection of the enriched dataset
training_df.limit(5).toPandas()


## 5. Validación de calidad antes de guardar

Antes de persistir el conjunto de entrenamiento en `Delta`, se realizan tres comprobaciones de calidad sobre el `DataFrame` materializado para detectar posibles anomalías que pudieran invalidar el entrenamiento:

* **Nulos en características**: se contabilizan los valores faltantes por columna. Un volumen inesperado podría indicar problemas en el `PiT` *join* o falta de cobertura histórica en las tablas de la capa `Gold`.
* **Consistencia del volumen**: se verifica que el número de filas del `DataFrame` enriquecido coincida exactamente con el de la *spine* original para asegurar que el cruce temporal no ha provocado pérdida de datos ni duplicidades por productos cartesianos.
* **Balance de clases**: se analiza la distribución de la variable objetivo (`is_fraud`) para cuantificar el desequilibrio natural del fraude y detectar posibles registros que aún carezcan de etiqueta.

In [0]:
feature_columns = [
    column for column in training_df.columns
    if column not in ["policy_id", "transaction_timestamp", "is_fraud"]
]

In [0]:
# Check 1: null count per feature column
null_counts_row = training_df.select([
    count(when(col(column).isNull(), column)).alias(column) for column in feature_columns
]).collect()[0]

for column in feature_columns:
    print(f"{column}: {null_counts_row[column]:,}")

In [0]:
# Check 2: row count consistency
spine_count = spine_df.count()
training_count = training_df.count()

print(f"Spine dataset rows: {spine_count:,}")
print(f"Training dataset rows: {training_count:,}")

In [0]:
# Check 3: class balance
print("Class balance:")
class_balance_rows = (
    training_df.groupBy("is_fraud")
               .count()
               .withColumn(
                   "pct", round(col("count") / training_count * 100, 2)
                )
               .orderBy("is_fraud").collect()
)

for row in class_balance_rows:
    value = "NULL" if row["is_fraud"] is None else row["is_fraud"]
    print(f"Label {value}: {row['count']:,} rows ({row['pct']}%)")

Los resultados obtenidos sobre nuestro conjunto de 5.94 millones de registros de siniestros arrojan información clave que valida la robustez de la arquitectura de Feature Store y la naturaleza real de los datos del sector asegurador:

**Consistencia del Volumen**
El conteo de filas resultante es idéntico al de la tabla spine de entrada (5,949,070 filas). Esto confirma que el proceso de Point-in-Time (PiT) Join ha funcionado de forma unívoca y precisa, reconstruyendo el estado del riesgo para cada siniestro sin generar duplicidades ni pérdida de registros.

**Naturaleza de los Valores Nulos**
Se observa una presencia significativa de nulos que responde a la lógica temporal de la arquitectura:

- **Perfil del Asegurado:** Las características estáticas (edad, tipo de vehículo, experiencia) presentan nulos en aproximadamente el 50% de los casos. Esto se debe a que el PiT Join actúa con rigor histórico: si un siniestro ocurrió en una fecha para la cual no existe todavía un registro de póliza válido en el historial SCD2 (Capa Silver), el sistema devuelve nulo. Esto garantiza que el modelo no se entrene con información "del futuro", respetando la validez temporal de los datos.

- **Métricas Comportamentales:** Los nulos en agregaciones (como el importe medio de los últimos 30 días) responden a una realidad estadística del negocio. Para pólizas nuevas o asegurados sin siniestralidad previa en la ventana de observación, las métricas de frecuencia e importe son indefinidas. La arquitectura mantiene esta integridad y delega la imputación de estos valores al pipeline de modelado.

**Balance de Clases y Maduración de Etiquetas**
Los datos reflejan fielmente el escenario de detección de fraude en seguros:

- **Desequilibrio de Clases:** Tras excluir los registros sin etiquetar, se observa un 3.97% de casos positivos (Fraude) frente a un 46.3% de casos legítimos. Este desequilibrio es el esperado en este dominio y es fundamental para configurar técnicas de pesado de clases (scale_pos_weight) en el algoritmo de entrenamiento.

- **Etiquetas Nulas (Casos en Maduración):** Se detecta un 49.72% de siniestros con etiqueta nula. Estos registros corresponden a operaciones que, bajo la lógica del negocio, aún no han sido auditadas o se encuentran en proceso de investigación. Al no contar con un resultado confirmado (Ground Truth), estos datos serán descartados durante la fase de entrenamiento para no introducir ruido en el aprendizaje del modelo, aunque representan un valioso conjunto para futuras inferencias una vez maduren.


## 6. Persistencia en `Delta` y registro de metadatos en `Unity Catalog`

Una vez validado el `DataFrame`, y tras **filtrar aquellas transacciones recientes que aún carecen de etiqueta (`is_fraud` nulo)**, se escribe en `Unity Catalog` como tabla `Delta` estática. El modo `overwrite` reemplaza el contenido de la tabla en cada ejecución, pero **no destruye el historial**: `Delta Lake` conserva todas las versiones anteriores de forma automática, lo que permite recuperar cualquier *snapshot* pasado mediante *time travel*.

Tras el guardado, se recupera el número de versión `Delta` resultante con `DESCRIBE HISTORY` y se registra como ***tag* de la tabla en `Unity Catalog`**. Esta es la forma correcta de asociar metadatos a un conjunto de datos: los *tags* pertenecen al dato y aseguran la trazabilidad exacta de qué versión histórica se ha generado en esta ejecución.

La propiedad `delta.enableChangeDataFeed` se activa en la tabla de salida para que, si en el futuro se configura un *pipeline* de monitorización incremental, pueda leer los cambios acumulados sin releer toda la tabla.

In [0]:
# Filter out recent transactions that lack a supervision label yet.
# A supervised learning algorithm cannot learn from unlabelled data.
clean_training_df = training_df.filter("is_fraud IS NOT NULL")
clean_training_count = clean_training_df.count()

# Persist the enriched training set as a static Delta table.
# Using overwrite so the table always reflects the latest generation run.
# Delta Lake automatically retains all previous versions for time travel.
(
    clean_training_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(gold_training_dataset_table)
)

print(f"Training dataset saved to: {gold_training_dataset_table}")

# Retrieve the Delta version number that was just written.
history_df = spark.sql(f"DESCRIBE HISTORY {gold_training_dataset_table}")
delta_version = history_df.select("version").first()[0]
print(f"Delta version written: {delta_version}")

In [0]:
generated_at = datetime.now(timezone.utc).isoformat()

# SET TAGS: surface metadata in the Catalog Explorer for human inspection.
# Tags belong to the data, not to a training run, and are visible
# in the Unity Catalog user interface but cannot be read via SHOW TBLPROPERTIES.
spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TAGS (
        'delta_version' = '{delta_version}',
        'spine_table' = '{gold_spine_table}',
        'feature_table_1' = '{gold_policy_profile_table}',
        'feature_table_2' = '{gold_policy_aggregations_table}',
        'label_col' = 'is_fraud',
        'num_rows' = '{clean_training_count}',
        'num_features' = '{len(feature_columns)}',
        'generated_at' = '{generated_at}'
    )
""")

# SET TBLPROPERTIES: persist the same metadata as table properties so that
# the training notebook can resolve the delta version programmatically
# via SHOW TBLPROPERTIES to keep a clean data → model traceability chain.
# The "ml." prefix avoids collisions with internal properties.
spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TBLPROPERTIES (
        'ml.delta_version' = '{delta_version}',
        'ml.spine_table' = '{gold_spine_table}',
        'ml.feature_table_1' = '{gold_policy_profile_table}',
        'ml.feature_table_2' = '{gold_policy_aggregations_table}',
        'ml.label_col' = 'is_fraud',
        'ml.num_rows' = '{clean_training_count}',
        'ml.num_features' = '{len(feature_columns)}',
        'ml.generated_at' = '{generated_at}'
    )
""")

# Add a human-readable description to the table
table_description = """
This managed table acts as the **static training dataset** for the machine learning model.
It is a point-in-time (`PiT`) snapshot joining the `gold_fraud_spine` with customer profiles
(`gold_customer_profile`) and rolling-window aggregations (`gold_customer_aggregations`),
pre-filtered to exclude unlabelled transactions.
"""
spark.sql(f"COMMENT ON TABLE {gold_training_dataset_table} IS '{table_description}'")

print(f"Unity Catalog metadata set on: {gold_training_dataset_table}")
print(f"Version: {delta_version}")
print(f"Number of rows: {clean_training_count:,}")
print(f"Number of features: {len(feature_columns)}")
print(f"Generated at: {generated_at}")
print()


## 7. Verificación del historial de versiones `Delta`

La última sección de esta libreta muestra el historial de operaciones de la tabla recién creada y ejemplifica cómo recuperar una instantánea exacta. En el ecosistema `Delta`, cada vez que se sobrescriben los datos o se modifican los metadatos, el motor registra internamente una nueva versión numérica incremental de forma automática. Esta capacidad de *time travel* es el estándar de la industria en entornos `MLOps`, ya que hace que los conjuntos de datos sean completamente inmutables y reproducibles a lo largo del tiempo.

A medida que esta libreta se ejecute en ciclos sucesivos, el historial acumulará nuevas versiones internas. Para conectar este proceso de ingeniería de datos con la fase de modelado de forma segura, los *tags* de `Unity Catalog` actúan como puente: se actualizan en cada ejecución para reflejar cuál es el número de la última versión de datos generada. De este modo, la libreta de entrenamiento consulta primero el *tag* de la tabla actual para saber qué versión numérica exacta debe recuperar. Al fijar la lectura a esa versión mediante *time travel* y registrar dicho número en el sistema de seguimiento de experimentos, se garantiza una cadena de trazabilidad inquebrantable desde los datos consumidos hasta el modelo final.

In [0]:
# Retrieve and display the internal version history.
# We dynamically extract all available metadata for each commit.
history_df = spark.sql(f"DESCRIBE HISTORY {gold_training_dataset_table}")
history_rows = history_df.collect()

for row in history_rows:
    for col_name in history_df.columns:
        val = row[col_name]
        # Skip nulls or empty dictionaries to keep the output clean and readable
        if val is not None and val != {} and val != "":
            print(f"{col_name}: {val}")
    print()

In [0]:
try:
    # Retrieve the latest table properties
    properties_df = spark.sql(f"SHOW TBLPROPERTIES {gold_training_dataset_table}")

    # Extract the Delta version using the "ml." prefix convention
    target_version_row = properties_df.filter("key = 'ml.delta_version'").first()

    if target_version_row:
        target_version = int(target_version_row["value"])
        print(f"Successfully retrieved property 'ml.delta_version' = {target_version}")
        print(f"Pinning data load to Delta version {target_version}")

        # Time travel: load the exact snapshot of data
        df_past = (
            spark.read
                 .format("delta")
                 .option("versionAsOf", target_version)
                 .table(gold_training_dataset_table)
        )
        print(f"Training dataset loaded successfully with {df_past.count():,} rows.")
    else:
        print("Warning: 'delta_version' tag not found on the table.")

except Exception as e:
    print(f"Failed to retrieve the version or load the data. Error: {e}")


## 8. Conclusiones y siguientes pasos

### ¿Qué hemos visto?

En esta libreta hemos construido el conjunto de datos de entrenamiento usando la `API` del `Feature Store` de `Databricks`:

1. La *spine* (`gold_fraud_spine`) define qué filas componen el conjunto de datos y en qué instante temporal se evalúa cada cruce. Es el punto de entrada obligatorio de `create_training_set`.
2. Los objetos `FeatureLookup` declaran cómo enriquecer cada fila de la *spine* con características de las tablas `Gold`. El parámetro `timestamp_lookup_key` activa el `PiT` *join*, que garantiza que cada fila recibe únicamente los valores de características disponibles en el momento exacto de la transacción, eliminando la posibilidad de fuga de datos del futuro.
3. `fe.create_training_set` construye el plan de ejecución y `.load_df()` materializa el `PiT` *join* de forma distribuida en `Spark`. El cruce se ejecuta una única vez y el resultado se guarda en `Delta`, de modo que los experimentos de hiperparámetros posteriores solo realizan lecturas secuenciales sobre la tabla estática.
4. Los metadatos del conjuntos de datos (versión `Delta`, tablas de origen (*spine* y características), columna objetivo, número de filas, número de características y fecha de generación) se registran como ***tags* y propiedades en `Unity Catalog`**. Esta es la separación de responsabilidades correcta: los metadatos pertenecen al dato; el sistema de seguimiento de experimentos recibirá el parámetro `delta_version` más adelante, cuando ya haya un modelo real al que enlazarlo.
5. Las comprobaciones de calidad (nulos, consistencia de volumen y balance de clases) permiten detectar problemas de cobertura en las tablas `Gold` antes de lanzar un ciclo de entrenamiento.

### ¿Cuándo volver a ejecutar esta libreta?

* **Primera vez**: para generar el conjunto de entrenamiento inicial y lanzar el primer ciclo de experimentación.
* **Cuando el sistema de monitorización detecte *drift* significativo**: el *job* de reentrenamiento automatizado lanzará esta libreta con datos más recientes y sobrescribirá la tabla `Delta`. `Delta Lake` conservará automáticamente la versión anterior en su historial.
* **Cuando se amplíe el conjunto de características**: si se añaden nuevas tablas de características a la capa `Gold`, habrá que actualizar los objetos `FeatureLookup` y regenerar el conjunto de datos.
* **Nunca de forma manual en producción**: esta libreta está diseñada para ejecutarse como tarea dentro de un *job* automatizado (por ejemplo, en `Lakeflow`). La ejecución manual solo tiene sentido durante el desarrollo.

### ¿Qué sigue?

Con el conjunto de entrenamiento disponible en `gold_fraud_training_dataset`, las dos líneas de trabajo que siguen pueden avanzar en paralelo:

1. **Fase de modelado**: con los datos listos, comienza la etapa de creación y entrenamiento de algoritmos. Las libretas de experimentación consumirán directamente la tabla estática `gold_fraud_training_dataset`. Durante estos ciclos, se consultarán las propiedades de la tabla para recuperar el valor numérico `delta_version` y registrarlo en el sistema de seguimiento de experimentos junto con los hiperparámetros probados y las métricas obtenidas. Esto asegurará la trazabilidad exacta entre el conjunto de datos consumido y los distintos modelos entrenados.

2. **Configuración del *baseline* de monitorización**: la tabla estática `gold_fraud_training_dataset` recién generada se usará como perfil de referencia (*baseline*) en la monitorización de calidad de datos. A medida que el modelo opere en producción, las características evaluadas en cada nueva transacción se registrarán en una tabla de inferencia (*inference table*), la cual comparte el mismo esquema exacto que nuestro conjunto de entrenamiento. Mediante la capacidad de `Data profiling`, el sistema calculará automáticamente las estadísticas de estas nuevas instancias y comparará la distribución de las características entrantes con la de nuestro *baseline* estático para identificar posibles desviaciones (*data drift*). Si el *drift* supera el umbral configurado, el sistema generará una alerta. Esta notificación permitirá al equipo evaluar si el cambio en el comportamiento de los datos justifica lanzar de nuevo esta libreta para generar una versión actualizada del conjunto de datos y comenzar un nuevo ciclo de reentrenamiento.